In [3]:
from langchain_community.vectorstores import FAISS
from langgraph.graph import START, END, StateGraph
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set
import pandas as pd

In [4]:
@dataclass
class Config:
    # Kaggle typically mounts competition data under /kaggle/input/<competition-name>/
    # For local testing, keep DATA_DIR='data'.
    DATA_DIR: str = 'data'

    # Reproducible artifact locations
    ARTIFACT_DIR: str = 'artifacts'
    SUBMISSION_NAME: str = 'submission.csv'

    # Corpus build controls
    # Default True to keep notebook stable under constrained memory.
    ENABLE_CHUNKING: bool = False
    CHUNK_CHARS: int = 1200
    OVERLAP_CHARS: int = 200

    # Dense retrieval stack
    OUT_K: int = 12
    STAGE_K: int = 150
    CITATION_SCORE_AGG: str = 'max'  # 'max' or 'mean'

    # Optional reranker
    USE_RERANKER: bool = False
    RERANKER_TOP_N: int = 50

    # Offline-safe model configuration
    # Dense model must be present locally for strict offline mode.
    DENSE_MODEL_NAME: str = 'intfloat/multilingual-e5-base'
    RERANKER_MODEL_NAME: str = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

    # Optional local model paths from Kaggle datasets, e.g. '/kaggle/input/my-models/e5-base'
    LOCAL_DENSE_MODEL_PATH: Optional[str] = None
    LOCAL_RERANKER_MODEL_PATH: Optional[str] = None

@dataclass
class DataPaths:
    root: Path

    @property
    def train(self) -> Path:
        return self.root / 'train.csv'

    @property
    def val(self) -> Path:
        return self.root / 'val.csv'

    @property
    def test(self) -> Path:
        return self.root / 'test.csv'

    @property
    def laws(self) -> Path:
        return self.root / 'laws_de.csv'

    @property
    def court(self) -> Path:
        return self.root / 'court_considerations.csv'

cfg = Config()

data_dir = Path(cfg.DATA_DIR)
artifact_dir = Path(cfg.ARTIFACT_DIR)
artifact_dir.mkdir(parents=True, exist_ok=True)

print(cfg)
print('Data dir exists:', data_dir.exists())

Config(DATA_DIR='data', ARTIFACT_DIR='artifacts', SUBMISSION_NAME='submission.csv', ENABLE_CHUNKING=False, CHUNK_CHARS=1200, OVERLAP_CHARS=200, OUT_K=12, STAGE_K=150, CITATION_SCORE_AGG='max', USE_RERANKER=False, RERANKER_TOP_N=50, DENSE_MODEL_NAME='intfloat/multilingual-e5-base', RERANKER_MODEL_NAME='cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', LOCAL_DENSE_MODEL_PATH=None, LOCAL_RERANKER_MODEL_PATH=None)
Data dir exists: True


In [5]:
# Data Cleaning
cfg = Config()
paths = DataPaths(root=Path(cfg.DATA_DIR))

law_df = pd.read_csv(paths.laws)
court_df = pd.read_csv(paths.court)
train_df = pd.read_csv(paths.train)
val_df = pd.read_csv(paths.val)
test_df = pd.read_csv(paths.test)




In [9]:
for row in law_df.itertuples(index=False):
    print(getattr(row, 'citation', ''))
    break

Art. 1 112


In [ ]:
# build corpus
def build_corpus(paths: DataPaths, cfg: Config) -> FAISS:
    """
    Build a unified corpus from the laws and court considerations datasets.
    """

    laws_df = pd.read_csv(paths.laws,usecols=['citation', 'text'])
    court_df = pd.read_csv(paths.court,usecols=['citation', 'text'])

    # Combine laws and court considerations into a single DataFrame
    combined_df = pd.concat([laws_df, court_df], ignore_index=True)
    combined_df.drop_duplicates(subset=['citation'], inplace=True)
    combined_df.reset_index(drop=True, inplace=True)
    print(f"Combined corpus size: {len(combined_df)}")

    




    
